# Inspect the HR source

Samples the change feed and the Germany parental-leave policy. Confirms the public records do not carry compensation or national identifiers.

In [1]:
import os

import httpx

BASE = os.environ.get("HARNESS_HR_URL", "http://127.0.0.1:8101")
FORBIDDEN = {"salary", "compensation", "ssn", "national_id"}

def keys(node):
    if isinstance(node, dict):
        return set(node) | {key for value in node.values() for key in keys(value)}
    if isinstance(node, list):
        return {key for value in node for key in keys(value)}
    return set()

sample = httpx.get(f"{BASE}/changes", params={"limit": 5}, timeout=30).json()
assert not keys(sample) & FORBIDDEN
item = sample["items"][0]
print(item["object_type"], item["id"], sorted(item["record"]))

policy = httpx.get(f"{BASE}/policies/pol-parental-leave-de", timeout=30).json()
print(policy["title"], policy["region"], policy["version"])
content = httpx.get(f"{BASE}/policies/pol-parental-leave-de/content", timeout=30)
assert "Elternzeit" in content.text
print(content.headers['content-type'], len(content.content), 'bytes')


person emp-000001 ['email', 'full_name', 'manager_id', 'native_acl', 'person_id', 'region', 'role', 'team', 'updated_at', 'version']
Parental Leave Policy (Germany) DE 1
text/markdown; charset=utf-8 2131 bytes
